# Predicting dispel4py cost with WfCommons

**The problem.** Someone asks "how long would this dispel4py workflow take with 500
tasks, and how much CPU and memory would it need?" Running it at 500 to find out
is expensive — and for an LLM-calling workflow, slow and billable.

**The approach.** Run it *small* a few times, learn what each PE costs per item,
generate a synthetic instance at the size you care about, and predict from that.

This notebook walks through `wfcommons.wfstream` on a real agentic
climate-sensor workflow. It predicts **runtime, CPU and memory, per PE and
overall**, and validates the predictions against runs that actually happened.

---
### What you need

1. dispel4py monitoring traces with resource metrics — the `monitor_*` artifacts
   from the `timed_simple` / `timed_multi` mappings, collected with resource
   sampling on. Set `TRACES` below.
2. This branch of WfCommons installed.

dispel4py itself is **not** needed: wfstream reads traces, it does not run
workflows.

## Setup

`TRACES` is the only path you must change. Everything else goes to a temporary
directory, except the cooked recipe — see the note in step 2.

In [1]:
import json, logging, pathlib, shutil, tempfile, warnings, collections

# >>> POINT THIS AT YOUR TRACES <<<
TRACES = pathlib.Path("/home/taina/dispel_data")

WORK = pathlib.Path(tempfile.mkdtemp(prefix="wfstream-demo-"))
WFFORMAT, BUILD, SYNTHETIC = WORK/"wfformat", WORK/"build", WORK/"synthetic"

# WfChef fits distributions per task type and is chatty about the ones that fail
warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)

print("traces:", TRACES)
print("work:  ", WORK)

traces: /home/taina/dispel_data
work:   /tmp/wfstream-demo-44lf1_g8


## Step 0 — the runs

The same workflow, traced under two mappings. `timed_simple` runs one process per
PE; `timed_multi` gives several PEs multiple ranks. **Both kinds matter, for
different reasons**, and we'll see why in steps 2 and 3.

dispel4py never records which real files the workflow read and wrote — the input
path arrives as a root input and the output path is a constant in the workflow
script — so we name them here.

In [2]:
TRACE_DIRS = ["monitoring_simple", "monitoring_multi_16", "monitoring_multi_32"]

INPUT_FILES = {
    "monitoring_simple":   ["sensor_data_agentic.json"],
    "monitoring_multi_16": ["sensor_data_parallel_100.json"],
    "monitoring_multi_32": ["sensor_data_parallel_100.json"],
}
OUTPUT_FILES = {
    "monitoring_simple":   ["agentic_sensor_results.jsonl"],
    "monitoring_multi_16": ["agentic_parallel_results.jsonl"],
    "monitoring_multi_32": ["agentic_parallel_results.jsonl"],
}

for d in TRACE_DIRS:
    runs = sorted((TRACES/d).glob("monitor_instances_run*.csv"))
    sizes = [sum(1 for _ in open(r)) - 1 for r in runs]
    print(f"{d:22} {len(runs)} run(s), {sizes} instances")

monitoring_simple      1 run(s), [7] instances
monitoring_multi_16    1 run(s), [16] instances
monitoring_multi_32    2 run(s), [17, 17] instances


## Step 1 — traces → WfFormat

One WfFormat **task per PE instance** (`pe_id@rank`), runtimes from the instances
CSV, edges from the concrete shape.

dispel4py is a streaming system: PEs exchange data in memory over named
connections, not through files. Each connection becomes a zero-byte WfFormat
"file" so the dependency survives the format. The real files bookend it.

The task count goes in the filename — step 2 depends on it.

In [3]:
from wfcommons.wfstream import convert_traces

instances = convert_traces.convert_traces(
    [TRACES/d for d in TRACE_DIRS], WFFORMAT,
    input_files=INPUT_FILES, output_files=OUTPUT_FILES)

for path in instances:
    print(convert_traces.describe(path))

monitoring_simple-7.json: 7 tasks, 7 streams, files=['sensor_data_agentic.json', 'agentic_sensor_results.jsonl']
monitoring_multi_16-16.json: 16 tasks, 18 streams, files=['sensor_data_parallel_100.json', 'agentic_parallel_results.jsonl']
monitoring_multi_32-17.json: 17 tasks, 18 streams, files=['sensor_data_parallel_100.json', 'agentic_parallel_results.jsonl']


## Step 2 — cook a recipe

WfChef finds **microstructures**: subgraphs that repeat as the workflow grows. It
finds them by *comparing instances of different sizes*.

Watch the output: the `simple` run contributes **0 microstructures**. A single
pipeline with one instance per PE has nothing that repeats. That is why one trace
is never enough.

**Installing** copies the recipe into the `wfcommons` package so it can be
imported. This writes inside your installed WfCommons; the last cell restores
it.

In [4]:
from wfcommons.wfstream import build_recipe

cooked = build_recipe.cook(WFFORMAT, BUILD, name="climate")
build_recipe.summarize(cooked)
print("installed into", build_recipe.install(cooked, name="climate"))

  monitoring_multi_16-16: 5 microstructure(s) ActionExecutorPE5x2, DecisionMergePE3x2, ParallelLLMSensorAgentPE4x4, DeterministicPrecheckPE2x3, NormalizeDataPE1x3
  monitoring_multi_32-17: 6 microstructure(s) ResultWriterPE6x2, ActionExecutorPE5x2, DecisionMergePE3x2, ParallelLLMSensorAgentPE4x4, DeterministicPrecheckPE2x3, NormalizeDataPE1x3
  monitoring_simple-7: 0 microstructure(s) 
  error table:
    ,monitoring_multi_16-16,monitoring_multi_32-17
    monitoring_simple-7,0.11415581486979587,0.11026144815768892
    monitoring_multi_16-16,0.0,0.024810764252159563
    monitoring_multi_32-17,,0.0
installed into /home/taina/miniconda3/envs/wfcommons/lib/python3.10/site-packages/wfchef_recipe_climate


### Making a cooked recipe discoverable

`create_recipe` writes a `pyproject.toml` at the build directory's root declaring
the recipe as a `workflow_recipes` entry point — but nothing installs it, so the
recipe stays invisible to `wfchef ls` and to `get_recipe`, which is how the rest
of WfCommons finds a recipe. On a fresh machine it simply is not there.

`register()` closes that gap by pip-installing the cooked package:

```python
build_recipe.register(BUILD)      # -> 'climate_recipe'
```

also available as `--register` on the CLI, or `on_new_workflow(..., register=True)`.
Once registered the recipe behaves like a built-in, and `load_recipe` resolves it
through its entry point rather than a hardcoded path — which is what makes it work
in a Colab session that only pip-installed the package.

This notebook uses `install()` instead, which skips pip and copies the data over
whichever copy of the recipe already resolves. Use `install()` to refresh a recipe
already in place, `register()` for one that is not.

Note that `install()` **overwrites that recipe's data** — the cell above prints
where. If you already had a `climate` recipe, it now holds this notebook's
version; re-cook or re-register to put yours back. The last cell says so too.

In [5]:
from wfcommons.wfstream import streaming_recipe

print("registered as an entry point:", build_recipe.registered("climate") is not None)
print("recipe data resolves to     :", streaming_recipe.recipe_path("climate"))

registered as an entry point: True
recipe data resolves to     : /home/taina/miniconda3/envs/wfcommons/lib/python3.10/site-packages/wfchef_recipe_climate


That second line matters: a recipe can exist in two places at once — registered in
site-packages and copied inside the wfcommons tree — and the two drift apart.
`recipe_path` derives the data directory from wherever the class was *actually*
loaded, and `install()` writes there, so the copy being read is always the copy
being updated.

## Step 3 — learn what each PE costs

This is separate from WfChef, which only learns runtime. `resource_stats` reads
the monitoring CSVs directly and learns **time, CPU and memory per PE**.

Three decisions are baked in, each forced by what the data actually looks like:

**Per-item costs, not per-run.** `total_secs` is the sum over ranks — aggregate
service time, independent of how many instances ran. Dividing by items gives a
primitive that is stable across runs of wildly different scale.

**CPU is learned on its own, never derived from runtime.** An LLM-calling PE
blocks on the network at ~10% CPU for ten seconds; a trivial PE runs at ~100% for
microseconds. Deriving one from the other is wrong in both directions.

**Memory is a baseline plus an attributable part.** RSS is ~71 MB of interpreter
before any PE does anything. The instrumentation's own metadata says *"never sum
across PEs or instances"*, so the model separates the shared floor from what each
PE adds on top.

In [6]:
from wfcommons.wfstream import resource_stats

stats = resource_stats.learn([TRACES/d for d in TRACE_DIRS])

print(f"baseline RSS {stats['baseline_rss_bytes']/1e6:.0f} MB, "
      f"widest run {stats['reference_items']:.0f} items\n")
print(f"{'PE':26} {'sel':>5} {'s/item':>9} {'cpu s/item':>11} {'cpu%':>6} {'mem MB':>8}")
for pe_id, pe in stats["pes"].items():
    mem = pe["rss_attributable_bytes"]
    print(f"{pe_id:26} {pe['selectivity']:>5.2f} {pe['secs_per_item']:>9.5f} "
          f"{pe['cpu_secs_per_item']:>11.6f} {pe['cpu_percent']:>6.1f} "
          f"{'n/a' if mem is None else f'{mem/1e6:>8.1f}'}")

baseline RSS 71 MB, widest run 100 items

PE                           sel    s/item  cpu s/item   cpu%   mem MB
ActionExecutorPE5           1.00   0.00049    0.000031   20.2      0.6
DecisionMergePE3            1.00   0.00021    0.000011   16.8      0.6
DeterministicPrecheckPE2    1.00   0.00027    0.000041   43.4      0.8
LLMSensorAgentPE4           0.05  10.79083    0.967952   10.3     27.6
NormalizeDataPE1            1.00   0.00025    0.000033   55.1      0.8
ResultWriterPE6             1.00   0.00077    0.000167   34.6      0.5
read0                       0.01   0.00874    0.005925   79.0      0.8


Two things to read off that table.

**Selectivity.** Items do not flow 1:1. `read0` makes one call; most PEs see every
item; the LLM PE sees **5%** of them, because a deterministic precheck filters
first. A simulator that assumed every stage sees every item would be twenty times
wrong on the most expensive stage.

**`LLMSensorAgentPE4` appears once, not twice.** The simple mapping calls it
`LLMSensorAgentPE4` and the parallel mapping calls it
`ParallelLLMSensorAgentPE4`, but it is the same PE doing the same work, so the
measurements pool.

In [7]:
print("aliases pooled into one PE:")
for pe_id, pe in stats["pes"].items():
    if len(pe["aliases"]) > 1:
        print(" ", pe_id, "<-", pe["aliases"])

llm = stats["pes"]["LLMSensorAgentPE4"]
print(f"\nper-item cost across {llm['runs_observed']} runs: "
      f"{llm['secs_per_item']:.2f}s "
      f"(min {llm['secs_per_item_min']:.2f}, max {llm['secs_per_item_max']:.2f}, "
      f"spread {llm['secs_per_item_max']/llm['secs_per_item_min']:.2f}x)")
print(f"idle instances excluded from the averages: {llm['idle_instances']}")

aliases pooled into one PE:
  LLMSensorAgentPE4 <- ['LLMSensorAgentPE4', 'ParallelLLMSensorAgentPE4']

per-item cost across 4 runs: 10.79s (min 8.56, max 14.51, spread 1.69x)
idle instances excluded from the averages: 3


That spread is the honest accuracy bound. Repeat runs of the *identical*
configuration differ by more than 1.5x, because the LLM API's latency varies that
much. No model can predict this workflow more precisely than the workflow
repeats, so predictions below carry the range rather than a single number.

The idle instances matter too: with 5 items over 4 ranks, one rank gets nothing.
It has no measurement window, so its CPU and memory columns are blank — and
averaging those blanks in as zeros would understate every rate.

## Step 4 — generate a synthetic instance

Here is the one place we **override WfChef**.

WfChef scales a workflow by replicating microstructures, and with none to hand it
copies the whole graph — reader included. For a streaming workflow that is wrong:
scaling a dispel4py workflow means giving PEs `numprocesses > 1`, not running the
pipeline twice.

So `grow_pipeline` starts from the **simple** run and replicates non-source PEs
round-robin. Sources are never replicated — dispel4py runs a source in exactly one
process. The simple run is found, not configured: it is the base graph with one
instance per PE.

Passing `stats` stamps the measured CPU and memory onto every generated task,
which WfCommons' generator otherwise leaves null.

In [8]:
from wfcommons.wfstream import generate_workflows, streaming_recipe

print("simple run:", streaming_recipe.simple_base_graph("climate"))

synthetic = generate_workflows.generate([120], SYNTHETIC, name="climate", stats=stats)[0]

execution = json.loads(synthetic.read_text())["workflow"]["execution"]["tasks"]
print("\na generated task carries its resource costs:")
for t in execution[:1] + [t for t in execution if t["id"].startswith("LLM")][:1]:
    print(" ", {k: t[k] for k in ("id", "runtimeInSeconds", "avgCPU", "memoryInBytes")})

simple run: monitoring_simple-7
  climate-120.json: 120 tasks, 141 streams, widths=[1, 20, 20, 20, 20, 20, 19]
    ActionExecutorPE5=20, DecisionMergePE3=20, DeterministicPrecheckPE2=20, LLMSensorAgentPE4=20, NormalizeDataPE1=20, ResultWriterPE6=19, read0=1
    sources: {'read0': 1}, components: 1

a generated task carries its resource costs:
  {'id': 'ActionExecutorPE5_00000001', 'runtimeInSeconds': 0.005, 'avgCPU': 20.22999947554245, 'memoryInBytes': 71564540}
  {'id': 'LLMSensorAgentPE4_00000004', 'runtimeInSeconds': 4.986, 'avgCPU': 10.339890948658844, 'memoryInBytes': 98515512}


Every generated instance is checked before it is kept: the metrics block must
match the real topology, and the graph must have **exactly one reader in exactly
one connected component**. Anything else is not a realisable dispel4py workflow,
so the file is deleted rather than left for something to train on.

## Step 5 — predict

An **analytical pipeline model**, not a task scheduler. dispel4py runs every PE
instance concurrently as its own process and streams items between them, so the
workflow does not end when a critical path ends — it ends when the slowest
*stage* has drained the stream:

```
items_p  = items x selectivity_p
stage_p  = ceil(items_p / instances_p) x secs_per_item_p
makespan = max_p stage_p + fill
```

Two details do the work.

**The ceiling.** Items are whole things, handed out one at a time, so a stage
lasts as long as its *busiest* instance. Five items across four instances go
2/1/1/1 — the stage costs two items, not the 1.25 an even split suggests. One
instance in the traced runs really did process zero items while another took the
extra.

**`fill` follows the graph.** The pipeline starts empty, so the first item must
reach the bottleneck and the last must traverse whatever follows it. That is the
slowest path *into* the bottleneck plus the slowest path *out of* it — not the
sum of every other stage. Two stages on parallel branches run at the same time,
so only the slower is charged, and a branch that bypasses the bottleneck is not
charged at all. The bottleneck's own per-item cost is already inside `stage_p`
and is never added again.

In [9]:
# the model against an exact discrete-event simulation of the same dataflow
import math, networkx as nx
from wfcommons.wfstream.simulate import fill_drain

def discrete_event(graph, cost, k, n):
    """Push n items through the graph, one instance at a time, and time it."""
    order = list(nx.topological_sort(graph))
    free = {p: [0.0]*k[p] for p in order}
    last = 0.0
    for _ in range(n):
        t = {}
        for p in order:
            ready = max((t[q] for q in graph.predecessors(p)), default=0.0)
            j = min(range(k[p]), key=lambda j: free[p][j])
            t[p] = max(ready, free[p][j]) + cost[p]
            free[p][j] = t[p]
        last = max(last, max(t.values()))
    return last

def analytical(graph, cost, k, n, use_graph=True):
    stage = {p: math.ceil(n/k[p])*cost[p] for p in graph}
    b = max(stage, key=lambda p: stage[p])
    f, _ = fill_drain(graph if use_graph else None, b, lambda p: cost[p], list(graph))
    return stage[b] + f

shapes = {
    "chain":          (nx.DiGraph([("s","a"),("a","B"),("B","c")]),
                       {"s":1,"a":1,"B":5,"c":1}),
    "bypass branch":  (nx.DiGraph([("s","B"),("s","side"),("B","m"),("side","m")]),
                       {"s":0.5,"B":5,"side":3,"m":0.5}),
    "parallel arms":  (nx.DiGraph([("s","x1"),("x1","x2"),("s","y1"),("y1","y2"),
                                   ("x2","B"),("y2","B")]),
                       {"s":0.5,"x1":1,"x2":1,"y1":2,"y2":2,"B":6}),
}
print(f"{'dataflow':16} {'exact':>8} {'sum of others':>15} {'critical path':>15}")
for name, (g, cost) in shapes.items():
    k = {p: 1 for p in g}
    print(f"{name:16} {discrete_event(g,cost,k,4):>8.2f} "
          f"{analytical(g,cost,k,4,False):>15.2f} {analytical(g,cost,k,4,True):>15.2f}")

dataflow            exact   sum of others   critical path
chain               23.00           23.00           23.00
bypass branch       21.00           24.00           21.00
parallel arms       28.50           30.50           28.50


Following the graph is exact. Summing every other stage is right only for a
chain, and overcharges the moment anything runs beside the bottleneck.

The model is a little optimistic when the bottleneck itself is replicated —
instance scheduling granularity costs slightly more than it charges.

In [10]:
from wfcommons.wfstream import simulate

prediction = simulate.simulate(synthetic, stats, items=1000)
print(simulate.report(prediction))

1000 items

PE                           inst    items    stage s     cpu s  cores    mem MB
--------------------------------------------------------------------------------
LLMSensorAgentPE4              20       50     32.372    48.398   2.07    1970.3
read0                           1       10      0.087     0.059   0.79      71.8
ResultWriterPE6                19     1000      0.041     0.167   6.57    1358.1
ActionExecutorPE5              20     1000      0.025     0.031   4.05    1431.3
DeterministicPrecheckPE2       20     1000      0.013     0.041   8.69    1434.1
NormalizeDataPE1               20     1000      0.012     0.033  11.03    1434.3
DecisionMergePE3               20     1000      0.011     0.011   3.35    1430.4

makespan            32.383 s   [25.7 - 43.5 across observed runs]
                               bottleneck LLMSensorAgentPE4 (32.372 s) + fill/drain 0.011 s (critical path)
cpu                 48.740 core-s  mean 1.51 cores, peak 36.54, util 1.3%
memory    

That is the working. `summary()` is the answer — the same prediction written for
someone who wants to know whether to run the thing.

The **Worth knowing** notes are conditional: each fires only when the prediction
actually rests on it. There are five — idle processes, memory the traces could
not attribute, startup cost assumed from a chain, PEs left out for lack of data,
and high run-to-run variance.

In [11]:
print(simulate.summary(prediction, "climate"))

climate: 120 processes, 1,000 items

  Runtime    32.4 seconds   (between 25.7 seconds and 43.5 seconds)
  CPU        1.5 cores on average, 49 core-seconds in total
  Memory     9.1 GB across 120 processes

  The time goes almost entirely to LLMSensorAgentPE4 (100% of it), which handles 50 of the 1,000 items across 20 processes.

  Worth knowing:
    - Processes are mostly idle -- 1.5 cores busy out of 120. Adding processes will not help unless LLMSensorAgentPE4 gets more of them.
    - The runs this was learned from varied by 1.7x among themselves, so treat the range as the answer rather than the single number.


## Does the model match reality?

Predict each real run from the pooled statistics and compare against what that run
measured. `stage_secs` is the busiest instance's wall time, so it is compared
against the largest `total_secs` among that PE's ranks.

In [12]:
import csv

def measured(csv_path):
    num = lambda v: float(v) if (v or "").strip() else 0.0
    rows = list(csv.DictReader(open(csv_path)))
    shape, secs, cpu = collections.Counter(), collections.defaultdict(list), collections.defaultdict(float)
    for r in rows:
        pe = resource_stats.canonical_pe(r["pe_id"])
        shape[pe] += 1
        secs[pe].append(num(r["total_secs"]))
        cpu[pe] += num(r["total_cpu_secs"])
    return shape, secs, cpu, int(max(sum(num(r["total_count"]) for r in rows if
        resource_stats.canonical_pe(r["pe_id"]) == pe) for pe in shape))

for run in sorted((TRACES/"monitoring_multi_32").glob("monitor_instances_run*.csv")):
    shape, secs, cpu, items = measured(run)
    result = simulate.simulate(shape, stats, items=items)
    print(f"--- {run.name.split('_run')[1][:22]}  ({items} items) ---")
    print(f"{'PE':24} {'pred s':>9} {'meas s':>9} | {'pred cpu':>9} {'meas cpu':>9}")
    for pe, p in sorted(result["pes"].items(), key=lambda kv: -kv[1]["stage_secs"])[:3]:
        print(f"{pe:24} {p['stage_secs']:>9.3f} {max(secs[pe]):>9.3f} | "
              f"{p['cpu_secs']:>9.3f} {cpu[pe]:>9.3f}")
    lo, hi = result["makespan_range_secs"]
    print(f"  makespan {result['makespan_secs']:.1f}s  [{lo:.1f} - {hi:.1f}]\n")

--- multi_20260924T0203544  (100 items) ---
PE                          pred s    meas s |  pred cpu  meas cpu
LLMSensorAgentPE4           21.582    15.832 |     4.840     5.341
ResultWriterPE6              0.039     0.042 |     0.017     0.022
ActionExecutorPE5            0.025     0.007 |     0.003     0.001
  makespan 21.6s  [17.1 - 29.0]

--- multi_20260924T0232371  (100 items) ---
PE                          pred s    meas s |  pred cpu  meas cpu
LLMSensorAgentPE4           21.582    31.055 |     4.840     6.320
ResultWriterPE6              0.039     0.055 |     0.017     0.015
ActionExecutorPE5            0.025     0.046 |     0.003     0.005
  makespan 21.6s  [17.1 - 29.0]



The bottleneck prediction lands between the two repeat runs, which is the best
outcome available: those two runs measured the same stage at 15.8 s and 31.1 s.
The cheap PEs are noisy in relative terms but they are five orders of magnitude
below the bottleneck, so they never move the makespan.

## How cost scales

Runtime grows with the stream; memory does not. Memory is a function of *shape* —
processes times baseline, plus each PE's footprint — which is what the traces
show, with RSS plateauing rather than accumulating per item.

In [13]:
print(f"{'items':>8} {'makespan s':>12} {'range':>20} {'core-s':>10} {'mem MB':>9}")
for n in (100, 1000, 10_000, 100_000):
    r = simulate.simulate(synthetic, stats, items=n)
    lo, hi = r["makespan_range_secs"]
    print(f"{n:>8} {r['makespan_secs']:>12.1f} {f'{lo:.0f} - {hi:.0f}':>20} "
          f"{r['cpu']['core_seconds']:>10.1f} {r['memory']['total_bytes']/1e6:>9.0f}")

   items   makespan s                range     core-s    mem MB
     100         10.8               9 - 15        4.9      9130
    1000         32.4              26 - 44       48.7      9130
   10000        269.8            214 - 363      487.4      9130
  100000       2697.7          2140 - 3627     4874.0      9130


## The two-call API

Everything above is one call. A registry decides which case applies and hands us
the trace directories.

**Case A — a workflow the registry has not seen.** Convert, cook, install, learn
costs, generate at the size the user asked about, predict.

In [14]:
from wfcommons.wfstream import on_new_workflow

result = on_new_workflow(
    [TRACES/d for d in TRACE_DIRS],
    num_tasks=500, items=5000, name="climate",
    wfformat_dir=WORK/"A"/"wfformat", build_dir=WORK/"A"/"build",
    synthetic_dir=WORK/"A"/"synthetic",
    input_files=INPUT_FILES, output_files=OUTPUT_FILES,
)
print("written:")
for key in ("synthetic", "stats", "prediction", "summary"):
    print(f"  {key:11} {result[key].name}")
print()
print(result["summary"].read_text())

  climate-500.json: 500 tasks, 584 streams, widths=[1, 83, 83, 83, 83, 84, 83]
    ActionExecutorPE5=84, DecisionMergePE3=83, DeterministicPrecheckPE2=83, LLMSensorAgentPE4=83, NormalizeDataPE1=83, ResultWriterPE6=83, read0=1
    sources: {'read0': 1}, components: 1
written:
  synthetic   climate-500.json
  stats       resource_stats.json
  prediction  climate-500.prediction.json
  summary     climate-500.summary.txt

climate: 500 processes, 5,000 items

  Runtime    43.2 seconds   (between 34.2 seconds and 58.0 seconds)
  CPU        5.6 cores on average, 244 core-seconds in total
  Memory     38.0 GB across 500 processes

  The time goes almost entirely to LLMSensorAgentPE4 (100% of it), which handles 250 of the 5,000 items across 83 processes.

  Worth knowing:
    - Processes are mostly idle -- 5.6 cores busy out of 500. Adding processes will not help unless LLMSensorAgentPE4 gets more of them.
    - The runs this was learned from varied by 1.7x among themselves, so treat the range 

A prediction that is only returned is gone when the process exits, so there is
nothing left to hold the eventual real run against. `*.prediction.json` keeps the
full result **and what produced it** — which instance, which statistics file,
what size was asked for, and when.

**Case B — known workflow, a new size was run for real.** Convert and store the
new run, re-cook, re-learn the costs, stop. Nothing is generated and nothing is
predicted: the point is that the next answer is better.

In [15]:
from wfcommons.wfstream import on_new_size_run

B = WORK/"B"
on_new_workflow([TRACES/d for d in TRACE_DIRS[:2]], num_tasks=40, name="climate",
                wfformat_dir=B/"wfformat", build_dir=B/"build",
                synthetic_dir=B/"synthetic", simulate=False,
                input_files=INPUT_FILES, output_files=OUTPUT_FILES)
print("corpus after registration:", sorted(p.name for p in (B/"wfformat").glob("*.json")))

r = on_new_size_run([TRACES/TRACE_DIRS[2]], name="climate",
                    wfformat_dir=B/"wfformat", build_dir=B/"build",
                    input_files=INPUT_FILES, output_files=OUTPUT_FILES)
print("added:", [p.name for p in r["added"]], "| re-cooked:", r["recipe"] is not None)

r2 = on_new_size_run([TRACES/TRACE_DIRS[2]], name="climate",
                     wfformat_dir=B/"wfformat", build_dir=B/"build",
                     input_files=INPUT_FILES, output_files=OUTPUT_FILES)
print("same run again ->", r2["skipped"], "| re-cooked:", r2["recipe"] is not None)

  climate-40.json: 40 tasks, 48 streams, widths=[1, 6, 7, 6, 7, 7, 6]
    ActionExecutorPE5=7, DecisionMergePE3=7, DeterministicPrecheckPE2=7, LLMSensorAgentPE4=6, NormalizeDataPE1=6, ResultWriterPE6=6, read0=1
    sources: {'read0': 1}, components: 1
corpus after registration: ['monitoring_multi_16-16.json', 'monitoring_simple-7.json']
added: ['monitoring_multi_32-17.json'] | re-cooked: True
same run again -> ['monitoring_multi_32'] | re-cooked: False


### The model scores itself

Case B does one more thing. Before a new run is folded into the statistics, the
model has **never seen it** — so predicting it at that moment is a genuine
held-out test, not a measure of how well the model fits data it already has.

Each result is appended to `accuracy.jsonl`, so the accuracy record accumulates
on its own as runs arrive, with no separate benchmarking step.

In [16]:
for check in r["accuracy"]:
    print(f"run {check['run_id'][:22]}  ({check['items']:.0f} items)")
    for pe, row in sorted(check["pes"].items(),
                          key=lambda kv: -(kv[1]["predicted_secs"] or 0))[:2]:
        print(f"  {pe:26} time {row['secs_ratio']:.2f}x   "
              f"cpu {row['cpu_ratio']:.2f}x")

log = B/"accuracy.jsonl"
print(f"\nappended to {log.name}: {len(log.read_text().strip().splitlines())} record(s)")

run multi_20260924T0203544  (100 items)
  LLMSensorAgentPE4          time 1.27x   cpu 0.72x
  ResultWriterPE6            time 0.67x   cpu 0.64x
run multi_20260924T0232371  (100 items)
  LLMSensorAgentPE4          time 0.65x   cpu 0.61x
  ResultWriterPE6            time 0.51x   cpu 0.94x

appended to accuracy.jsonl: 2 record(s)


Both ratios bracket 1.0 — the model lands between the two runs, which is the most
that can be asked when those runs differ from *each other* by 2x.

Only per-stage figures can be scored: dispel4py records service time per PE and
never the workflow's wall clock, so the predicted makespan has nothing to be
compared against. The bottleneck stage is the closest proxy.

## Limits worth knowing

- **Memory does not scale with items.** It is modelled from shape alone. A PE that
  accumulates state per item would break that assumption.
- **No end-to-end wall clock is recorded** in the traces, only per-PE service
  time. The makespan model is validated per stage, not against a measured
  makespan.
- **Selectivity is learned from the widest run, as a ratio.** In these traces the
  precheck passed 5 items out of both 6 and 100 — a constant *count*, not a
  constant fraction. Extrapolating it as a fraction is the least certain part of
  the model, and it lands on the stage that dominates. A run over input with a
  different anomaly count would settle it.
- **Feedback loops** are kept as an iteration count per PE rather than as a cycle,
  so WfChef still sees a DAG. Untested against a real looped workflow.
- **Variance bounds precision.** For this workflow the LLM call varies ~1.7x
  between identical runs, so the range matters more than the point estimate.

## Module map

| Module | Role |
|---|---|
| `dispel_fwd_converter` | traces → WfFormat (the parser) |
| `convert_traces` | rebuild the corpus from a set of traces |
| `update_traces` | add new runs to an existing corpus |
| `build_recipe` | cook + install a WfChef recipe |
| `streaming_recipe` | the scaling rule: replicate PEs, not pipelines |
| `generate_workflows` | write synthetic instances, validate, stamp resources |
| `resource_stats` | per-PE time / CPU / memory learned from the CSVs |
| `simulate` | the prediction, its summary, and scoring it against a run |
| `pipeline` | `on_new_workflow` / `on_new_size_run` |
| `config` | defaults for the climate workflow |

Each is a CLI too:

```bash
python -m wfcommons.wfstream.resource_stats monitoring_* -o stats.json
python -m wfcommons.wfstream.simulate climate-500.json -s stats.json -i 5000
```

## Cleanup

Removes the temporary directory, and restores the recipe if it lives inside the
wfcommons tree. A recipe that was pip-installed cannot be restored by git — the
cell says what to do instead.

In [17]:
import subprocess

shutil.rmtree(WORK, ignore_errors=True)
print("removed", WORK)

installed_into = streaming_recipe.recipe_path("climate")
repo = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "examples" else pathlib.Path.cwd()
if str(installed_into).startswith(str(repo.resolve())):
    subprocess.run(["git", "checkout", "--", "wfcommons/wfchef/recipes/"], cwd=repo)
    print("restored the in-tree recipe at", installed_into)
else:
    print(f"the climate recipe at {installed_into} now holds this notebook's "
          f"version.\n  To put your own back: re-cook it and call "
          f"build_recipe.register(<your build dir>),\n  or "
          f"`wfchef uninstall climate` to drop it entirely.")

removed /tmp/wfstream-demo-44lf1_g8
the climate recipe at /home/taina/miniconda3/envs/wfcommons/lib/python3.10/site-packages/wfchef_recipe_climate now holds this notebook's version.
  To put your own back: re-cook it and call build_recipe.register(<your build dir>),
  or `wfchef uninstall climate` to drop it entirely.
